In [0]:
%pip install mlflow>=3.0 databricks-feature-engineering>=0.13.0
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 103.1 MB/s eta 0:00:00
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.4.0
    Uninstalling mlflow-skinny-3.4.0:
      Successfully uninstalled mlflow-skinny-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
databricks-serverless-gpu 0.5.9 requires databricks-connect<16,>=15.4.2, but you have databricks-connect 17.2.4 which is incompatible.
databricks-serverless-gpu 0.5.9 requires mlflow<3.0,>=2.17, but you have mlflow 3.8.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType

# 1) Load Silver v2
df = spark.table("mlops_project.lendingclub_silver_v2")
df = df.drop("emp_length", "title")


label_col = "label_default"

# 2) Drop non-feature cols if present (keep label)
drop_non_features = ["_ingest_ts", "_source_file"]
for c in drop_non_features:
    if c in df.columns:
        df = df.drop(c)

# 3) Basic sanity
df.select(label_col).groupBy(label_col).count().orderBy(label_col).show()

# 4) Identify feature columns by type
numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != label_col
]

categorical_cols = [
    f.name for f in df.schema.fields
    if f.dataType.simpleString() == "string"
]

print("Numeric:", len(numeric_cols))
print("Categorical:", len(categorical_cols))
print("Example categoricals:", categorical_cols[:15])

# 5) Train/validation/test split
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)

print("Train rows:", train_df.count())
print("Val rows:", val_df.count())
print("Test rows:", test_df.count())

# quick label sanity (optional)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name)
    d.groupBy("label_default").count().orderBy("label_default").show()


+-------------+-------+
|label_default|  count|
+-------------+-------+
|            0|1041952|
|            1| 287320|
+-------------+-------+

Numeric: 88
Categorical: 17
Example categoricals: ['term', 'grade', 'sub_grade', 'home_ownership', 'verification_status', 'pymnt_plan', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'verification_status_joint', 'hardship_flag', 'hardship_type', 'hardship_status', 'disbursement_method']
Train rows: 930493
Val rows: 199473
Test rows: 199306
train
+-------------+------+
|label_default| count|
+-------------+------+
|            0|729037|
|            1|201256|
+-------------+------+

val
+-------------+------+
|label_default| count|
+-------------+------+
|            0|156817|
|            1| 43177|
+-------------+------+

test
+-------------+------+
|label_default| count|
+-------------+------+
|            0|156098|
|            1| 42887|
+-------------+------+



In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("emp_length")

(
    df.groupBy("emp_length", "label_default")
      .count()
      .withColumn(
          "proportion",
          F.col("count") / F.sum("count").over(w)
      )
      .orderBy("emp_length", F.desc("proportion"))
      .show(truncate=False)
)

+----------+-------------+------+-------------------+
|emp_length|label_default|count |proportion         |
+----------+-------------+------+-------------------+
|1 year    |0            |67986 |0.7770539020710465 |
|1 year    |1            |19506 |0.2229460979289535 |
|10+ years |0            |347620|0.7965317574704927 |
|10+ years |1            |88797 |0.20346824252950732|
|2 years   |0            |94343 |0.7852166892774805 |
|2 years   |1            |25806 |0.21478331072251955|
|3 years   |0            |83267 |0.7832397400080895 |
|3 years   |1            |23044 |0.21676025999191054|
|4 years   |0            |62507 |0.7853230142975601 |
|4 years   |1            |17087 |0.21467698570243987|
|5 years   |0            |65546 |0.7877556906953825 |
|5 years   |1            |17660 |0.21224430930461746|
|6 years   |0            |49075 |0.791315284518761  |
|6 years   |1            |12942 |0.208684715481239  |
|7 years   |0            |46763 |0.7920025743513313 |
|7 years   |1            |12

In [0]:
# free previous fitted models if they exist
for name in ["preprocess_model", "preprocess_pipeline", "train_prepared", "val_prepared", "test_prepared"]:
    if name in globals():
        del globals()[name]

import gc
gc.collect()

3

In [0]:
(train_df.count(), len(train_df.columns))

(930493, 106)

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, FeatureHasher
from pyspark.sql import functions as F

label_col = "label_default"

# --- 1) Remove leakage-ish / operational categoricals (keep it simple & defensible)
drop_feature_cols = [c for c in df.columns if c.startswith("hardship_")] + ["pymnt_plan"]
categorical_cols_v2 = [c for c in categorical_cols if c not in drop_feature_cols]

print("Dropped categoricals:", [c for c in categorical_cols if c in drop_feature_cols])
print("Categoricals used:", len(categorical_cols_v2))

# --- 2) Numeric imputation
imputed_numeric_cols = [f"{c}__imputed" for c in numeric_cols]
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=imputed_numeric_cols,
    strategy="median"
)

# 3) Hash everything into a fixed-size vector
# numFeatures controls dimensionality; 2^18 or 2^19 is a good start
hasher = FeatureHasher(
    inputCols=categorical_cols_v2 + imputed_numeric_cols,
    outputCol="features",
    numFeatures=2**18
)

preprocess_pipeline = Pipeline(stages=[imputer, hasher])


# Fit on TRAIN only (best practice)
preprocess_model = preprocess_pipeline.fit(train_df.sample(fraction=0.1, seed=42))

train_prepared = preprocess_model.transform(train_df).select(label_col, "features")
val_prepared   = preprocess_model.transform(val_df).select(label_col, "features")
test_prepared  = preprocess_model.transform(test_df).select(label_col, "features")

print("Categoricals hashed:", len(categorical_cols_v2))
print("Prepared cols:", train_prepared.columns)
train_prepared.limit(3).show(truncate=False)


Dropped categoricals: ['pymnt_plan', 'hardship_flag', 'hardship_type', 'hardship_status']
Categoricals used: 13
Categoricals hashed: 13
Prepared cols: ['label_default', 'features']
+-------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
import mlflow
from mlflow import spark as mlflow_spark

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from mlflow.models.signature import infer_signature

# Set the MLflow registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")


label_col = "label_default"

# ----------------------------
# 1) Add class weights (recommended for imbalance)
# ----------------------------
counts = train_prepared.groupBy(label_col).count().collect()
cnt = {row[label_col]: row["count"] for row in counts}
n0, n1 = cnt.get(0, 0), cnt.get(1, 0)
total = n0 + n1

# weights: give minority class higher weight
w0 = total / (2.0 * n0) if n0 else 1.0
w1 = total / (2.0 * n1) if n1 else 1.0

train_w = train_prepared.withColumn(
    "class_weight",
    F.when(F.col(label_col) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

val_w  = val_prepared.withColumn("class_weight", F.lit(1.0))
test_w = test_prepared.withColumn("class_weight", F.lit(1.0))

print("Class counts:", cnt, "weights:", {"w0": w0, "w1": w1})

# ----------------------------
# 2) Scaler + Logistic Regression
# ----------------------------
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=False  # must be False for sparse vectors
)

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol=label_col,
    weightCol="class_weight",
    maxIter=50,
    regParam=0.0,     # we can tune later
    elasticNetParam=0.0
)

model_pipeline = Pipeline(stages=[scaler, lr])

# ----------------------------
# 3) Train + Evaluate (Validation)
# ----------------------------
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)



Class counts: {1: 201256, 0: 729037} weights: {'w0': 0.6380286597250894, 'w1': 2.3112180506419686}


In [0]:
import os
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 0) UC temp dir required on serverless/shared
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"

# 1) Pick a stable experiment + model name
mlflow.set_experiment("/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps")
model_name = "lr_baseline_featurehasher_model"


with mlflow.start_run(run_name="lr_baseline_featurehasher") as run:
    # 1) Train -> this is where "fitted" comes from
    fitted = model_pipeline.fit(train_w.sample(fraction=0.5, seed=42))

    # 2) Validate
    val_pred = fitted.transform(val_w)
    auc = evaluator_auc.evaluate(val_pred)
    pr_auc = evaluator_pr.evaluate(val_pred)

    # 3) Signature + input example
    # Use a small pandas sample from the *prepared* input (because that's what the model consumes)
    input_example_pd = train_w.select("features", "class_weight", label_col).limit(20).toPandas()
    pred_example_pd = val_pred.select("probability").limit(20).toPandas()
    signature = infer_signature(input_example_pd, pred_example_pd)
    model_name = "lr_baseline_featurehasher_model"

    # 4) Log params/metrics
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("encoding", "FeatureHasher")
    mlflow.log_param("numFeatures_hash", 2**18)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("regParam", 0.0)
    mlflow.log_param("elasticNetParam", 0.0)
    mlflow.log_param("class_weighting", True)
    mlflow.log_metric("val_auc", float(auc))
    mlflow.log_metric("val_pr_auc", float(pr_auc))

    # 5) Log + register Spark model
    mlflow.spark.log_model(
        spark_model=fitted,
        artifact_path="model",
        #input_example=input_example_pd,
        signature=signature,
        registered_model_name=model_name
    )

print("Validation AUC:", auc)
print("Validation PR AUC:", pr_auc)
print(f"Registered model name: {model_name}")

print(f"Model '{model_name}' has been registered with MLflow.")


2026/01/01 14:47:51 INFO mlflow.tracking.fluent: Experiment with name '/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps' does not exist. Creating a new experiment.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-ee5c94b7-dbde-4e63-94c9-e645c47067eb/lib/python3.12/site-packages/mlflow/system_metrics/metrics/gpu_monitor.py:11: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
2026/01/01 14:47:52 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/01/01 14:49:24 WARNING mlflow.models.signature: Failed to infer schema for inputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. Note that MLflow doesn't validate data types during inference for AnyType. To see the full traceback, set logging level to DEBUG.
2026/01/01 14:49:24 WARNING mlflow.models.signature: F

Uploading artifacts:   0%|          | 0/28 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.lr_baseline_featurehasher_model': https://dbc-8dfd01aa-fa50.cloud.databricks.com/explore/data/models/workspace/default/lr_baseline_featurehasher_model/version/1?o=7474657027313099
2026/01/01 14:49:47 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/01 14:49:47 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Validation AUC: 0.7537620840458397
Validation PR AUC: 0.5302639053165927
Registered model name: lr_baseline_featurehasher_model
Model 'lr_baseline_featurehasher_model' has been registered with MLflow.


In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F

model_name = "lr_baseline_featurehasher_model"
model_uri = f"models:/{model_name}@latest"  

model = mlflow.spark.load_model(model_uri)


In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

scored = (
    model.transform(test_prepared)
         .withColumn("prob_array", vector_to_array("probability"))
         .withColumn("default_proba", F.col("prob_array")[1])
         .withColumn("prediction_ts", F.current_timestamp())
         .withColumn("model_name", F.lit(model_name))
         .drop("prob_array")
)

scored.select("default_proba", "label_default", "prediction_ts", "model_name").show(5)

+-------------------+-------------+--------------------+--------------------+
|      default_proba|label_default|       prediction_ts|          model_name|
+-------------------+-------------+--------------------+--------------------+
|0.10256641038063807|            0|2026-01-01 15:11:...|lr_baseline_featu...|
|0.40592038414752774|            1|2026-01-01 15:11:...|lr_baseline_featu...|
|0.28629546788965976|            0|2026-01-01 15:11:...|lr_baseline_featu...|
| 0.2661732211274822|            0|2026-01-01 15:11:...|lr_baseline_featu...|
| 0.1823871304412904|            0|2026-01-01 15:11:...|lr_baseline_featu...|
+-------------------+-------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
(scored
    .select(
        "default_proba",
        "label_default",
        "model_name",
        "prediction_ts"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mlops_project.lendingclub_gold_predictions")
)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    labelCol="label_default",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

test_auc = evaluator.evaluate(scored)
print("TEST AUC:", test_auc)

TEST AUC: 0.7541607955787182


End to End Pipeline

In [0]:
import os
import mlflow
import mlflow.spark

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, FeatureHasher, StandardScaler
from mlflow.models.signature import infer_signature
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# -------------------------
# Config
# -------------------------
label_col = "label_default"
model_name = "lr_end_to_end_featurehasher_model"
experiment_path = "/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps"

# UC temp dir needed on serverless/shared
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"

mlflow.set_experiment(experiment_path)

# -------------------------
# Load data (raw Silver_v2)
# -------------------------
df = spark.table("mlops_project.lendingclub_silver_v2")

# Splits (same idea as before)
train_df, val_df, test_df = df.randomSplit([0.7, 0.15, 0.15], seed=42)

# Identify feature columns
numeric_cols = [
    c for c, t in df.dtypes
    if t in ("int", "bigint", "double", "float") and c != label_col
]
categorical_cols = [c for c, t in df.dtypes if t == "string"]

# -------------------------
# 1) Remove leakage-ish categoricals (EXACTLY like your code)
# -------------------------
drop_feature_cols = [c for c in df.columns if c.startswith("hardship_")] + ["pymnt_plan","emp_length", "title"]
categorical_cols_v2 = [c for c in categorical_cols if c not in drop_feature_cols]

print("Dropped categoricals:", [c for c in categorical_cols if c in drop_feature_cols])
print("Categoricals used:", len(categorical_cols_v2))

# -------------------------
# 2) Numeric imputation (EXACTLY like your code)
# -------------------------
imputed_numeric_cols = [f"{c}__imputed" for c in numeric_cols]
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=imputed_numeric_cols,
    strategy="median"
)

# -------------------------
# 3) Feature hashing (EXACTLY like your code)
# -------------------------
hasher = FeatureHasher(
    inputCols=categorical_cols_v2 + imputed_numeric_cols,
    outputCol="features",
    numFeatures=2**18
)

# -------------------------
# 4) Add scaling + Logistic Regression (the "model part")
# -------------------------
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withStd=True,
    withMean=False  # required for sparse vectors
)

lr = LogisticRegression(
    featuresCol="features_scaled",
    labelCol=label_col,
    weightCol="class_weight",
    maxIter=50,
    regParam=0.0,
    elasticNetParam=0.0
)

full_pipeline = Pipeline(stages=[imputer, hasher, scaler, lr])

# -------------------------
# 5) Class weights (same concept as before)
# -------------------------
counts = train_df.groupBy(label_col).count().collect()
cnt = {r[label_col]: r["count"] for r in counts}
n0, n1 = cnt.get(0, 0), cnt.get(1, 0)
total = n0 + n1

w0 = total / (2.0 * n0) if n0 else 1.0
w1 = total / (2.0 * n1) if n1 else 1.0

train_w = train_df.withColumn(
    "class_weight",
    F.when(F.col(label_col) == 1, F.lit(w1)).otherwise(F.lit(w0))
)

# -------------------------
# 6) Fit end-to-end model with preprocessing fit-on-sample (EXACT idea)
# IMPORTANT: We fit the whole pipeline on a sampled TRAIN set to avoid Spark Connect size limits.
# -------------------------
fit_fraction = 0.1
train_fit_df = train_w.sample(fraction=fit_fraction, seed=42)

# Evaluator
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

with mlflow.start_run(run_name="lr_end_to_end_featurehasher") as run:
    full_fitted = full_pipeline.fit(train_fit_df)

    # Validate on full validation set (no weighting needed for scoring)
    val_scoring = val_df.withColumn("class_weight", F.lit(1.0))
    val_pred = full_fitted.transform(val_scoring)
    val_auc = evaluator_auc.evaluate(val_pred)

    input_example_pd = train_fit_df.drop("class_weight").limit(20).toPandas()
    pred_example_pd = val_pred.select("probability", "prediction").limit(20).toPandas()
    signature = infer_signature(input_example_pd, pred_example_pd)

    # Log key params/metrics
    mlflow.log_param("encoding", "FeatureHasher")
    mlflow.log_param("numFeatures_hash", 2**18)
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("fit_fraction", fit_fraction)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("regParam", 0.0)
    mlflow.log_param("elasticNetParam", 0.0)
    mlflow.log_param("class_weighting", True)
    mlflow.log_metric("val_auc", float(val_auc))

    # Register end-to-end model
    mlflow.spark.log_model(
        spark_model=full_fitted,
        artifact_path="model",
          signature=signature,
        registered_model_name=model_name
    )

print("Registered end-to-end model:", model_name)
print("Validation AUC:", val_auc)


2026/01/04 09:59:40 INFO mlflow.tracking.fluent: Experiment with name '/Users/angelina.suchkova@mastercard.com/LendingClub_MLOps' does not exist. Creating a new experiment.


Dropped categoricals: ['emp_length', 'pymnt_plan', 'hardship_flag', 'hardship_type', 'hardship_status']
Categoricals used: 13


/local_disk0/.ephemeral_nfs/envs/pythonEnv-164791cc-8151-4d95-be91-4a5848177d5a/lib/python3.12/site-packages/mlflow/system_metrics/metrics/gpu_monitor.py:11: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml
2026/01/04 09:59:47 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-164791cc-8151-4d95-be91-4a5848177d5a/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) th

Uploading artifacts:   0%|          | 0/40 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.lr_end_to_end_featurehasher_model': https://dbc-8dfd01aa-fa50.cloud.databricks.com/explore/data/models/workspace/default/lr_end_to_end_featurehasher_model/version/1?o=7474657027313099
2026/01/04 10:01:17 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/04 10:01:17 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Registered end-to-end model: lr_end_to_end_featurehasher_model
Validation AUC: 0.752265778669991
